In [1]:
from datetime import datetime
import copy
import json
import random
import re
import uuid

In [2]:
legacy_systems = {

    "payroll": {
        1001: {
            "salary": 85000
        },
        1002: {
            "salary": 92000
        }
    },

    "benefits": {
        1001: {
            "benefits": "Premium Plan"
        },
        1002: {
            "benefits": "Standard Plan"
        }
    },

    "hr": {
        1001: {
            "department": "Engineering"
        },
        1002: {
            "department": "Finance"
        }
    }
}

In [3]:
snapshot = copy.deepcopy(legacy_systems)

def restore_snapshot():

    global legacy_systems

    legacy_systems = copy.deepcopy(snapshot)

    print("[SELF-HEAL] Snapshot restored")

In [4]:
policies = {

    "Angela": {

        "employee_id": 1001,

        "allowed_outcomes": [
            "salary",
            "benefits",
            "compensation_summary"
        ],

        "max_requests": 5
    },

    "Bob": {

        "employee_id": 1002,

        "allowed_outcomes": [
            "salary"
        ],

        "max_requests": 3
    }
}

In [5]:
audit_log = []

request_counts = {
    user: 0
    for user in policies
}

trust_scores = {
    user: 100
    for user in policies
}

In [6]:
def log_event(user, action, result):

    audit_log.append({
        "timestamp":
            datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
        "user": user,
        "action": action,
        "result": result
    })

In [7]:
def validate_system():

    for emp_id in legacy_systems["payroll"]:

        salary = legacy_systems["payroll"][emp_id]["salary"]

        if salary <= 0:

            print(
                f"[ANOMALY] Invalid salary for {emp_id}"
            )

            restore_snapshot()

            return False

    return True

In [8]:
def check_trust(user):

    score = trust_scores.get(
        user,
        0
    )

    print(
        f"[TrustNode] Score = {score}"
    )

    return score > 0

In [9]:
def check_rate_limit(user):

    limit = policies[user]["max_requests"]

    if request_counts[user] >= limit:

        return False

    request_counts[user] += 1

    return True

In [10]:
def PolicyNode(user, outcome):

    print(
        "[PolicyNode] Policy passed"
    )

    return True


def TrustNode(user, outcome):

    return check_trust(user)


def RecoveryNode(user, outcome):

    print(
        "[RecoveryNode] Integrity verified"
    )

    return validate_system()


def AuditNode(user, outcome):

    print(
        "[AuditNode] Request logged"
    )

    return True


def ComplianceNode(user, outcome):

    print(
        "[ComplianceNode] Compliance passed"
    )

    return True

In [11]:
node_map = {

    "PolicyNode": PolicyNode,

    "TrustNode": TrustNode,

    "RecoveryNode": RecoveryNode,

    "AuditNode": AuditNode,

    "ComplianceNode": ComplianceNode
}

In [12]:
validation_nodes = list(
    node_map.keys()
)

def build_route():

    return random.sample(
        validation_nodes,
        random.randint(
            2,
            len(validation_nodes)
        )
    )

In [13]:
active_conduits = {}

def create_conduit(user):

    conduit_id = str(
        uuid.uuid4()
    )[:8]

    active_conduits[conduit_id] = {
        "user": user
    }

    return conduit_id


def destroy_conduit(conduit_id):

    if conduit_id in active_conduits:

        del active_conduits[
            conduit_id
        ]

In [14]:
def salary_outcome(user):

    employee_id = policies[user][
        "employee_id"
    ]

    return {
        "salary":
            legacy_systems[
                "payroll"
            ][employee_id][
                "salary"
            ]
    }


def benefits_outcome(user):

    employee_id = policies[user][
        "employee_id"
    ]

    return {
        "benefits":
            legacy_systems[
                "benefits"
            ][employee_id][
                "benefits"
            ]
    }


def compensation_summary(user):

    employee_id = policies[user][
        "employee_id"
    ]

    return {

        "salary":
            legacy_systems[
                "payroll"
            ][employee_id][
                "salary"
            ],

        "benefits":
            legacy_systems[
                "benefits"
            ][employee_id][
                "benefits"
            ],

        "department":
            legacy_systems[
                "hr"
            ][employee_id][
                "department"
            ]
    }

In [15]:
outcomes = {

    "salary":
        salary_outcome,

    "benefits":
        benefits_outcome,

    "compensation_summary":
        compensation_summary
}


In [16]:
def request_outcome(
    user,
    outcome
):

    if user not in policies:

        return {
            "error":
                "Unknown user"
        }

    if outcome not in outcomes:

        return {
            "error":
                "Unknown outcome"
        }

    if outcome not in policies[user][
        "allowed_outcomes"
    ]:

        return {
            "error":
                "Outcome denied"
        }

    if not check_rate_limit(user):

        return {
            "error":
                "Rate limit exceeded"
        }

    conduit_id = create_conduit(
        user
    )

    route = build_route()

    print(
        "\n========== ROUTE =========="
    )

    print(
        f"Conduit: {conduit_id}"
    )

    for node_name in route:

        print(
            "→",
            node_name
        )

        passed = node_map[
            node_name
        ](
            user,
            outcome
        )

        if not passed:

            destroy_conduit(
                conduit_id
            )

            return {
                "error":
                    f"Blocked by {node_name}"
            }

    result = outcomes[
        outcome
    ](
        user
    )

    log_event(
        user,
        outcome,
        "ALLOWED"
    )

    destroy_conduit(
        conduit_id
    )

    return result

In [17]:
outcome_catalog = {

    "salary":
        "Employee salary information",

    "benefits":
        "Employee benefits package",

    "compensation_summary":
        "Combined payroll, benefits and HR information"
}

for name, description in outcome_catalog.items():

    print(
        f"{name} -> {description}"
    )

salary -> Employee salary information
benefits -> Employee benefits package
compensation_summary -> Combined payroll, benefits and HR information


In [18]:
print(
    request_outcome(
        "Angela",
        "compensation_summary"
    )
)


========== ROUTE ==========
Conduit: adad601e
→ PolicyNode
[PolicyNode] Policy passed
→ ComplianceNode
[ComplianceNode] Compliance passed
→ RecoveryNode
[RecoveryNode] Integrity verified
→ AuditNode
[AuditNode] Request logged
→ TrustNode
[TrustNode] Score = 100
{'salary': 85000, 'benefits': 'Premium Plan', 'department': 'Engineering'}
